# Shape the Shop: Safe Execution Hybrid

The previous four-route, prefix-safe programme remains the strategic backbone. This revision imports only execution-safety ideas from the five-route chassis and keeps the lightweight bot's useful principle: route decisions remain sparse and checkpoint-based. No new production branch is introduced without a validated continuation.


## Backbone and branch discipline

For active route $R_j$ and candidate continuation $R_k$ at checkpoint $\tau$, switching remains legal only when

$$
R_j(t)=R_k(t)
\qquad\text{for every }t<\tau.
$$

The existing checkpoints at turns 226, 360, and 433 are unchanged. Consequently, the richer but unverified tape-splicing behavior of the lightweight portfolio is not imported into the release candidate.


## Block affordability guard

At every 72-turn boundary, let $C$ be the planned cost of hires, land, seeds, products, and animals in the next block. Let $M$ be current money and let $q_i$ be the quantity of product $i$ already scheduled for sale inside the block. The projected coverage is

$$
F=M+\sum_i p_i\min(s_i,q_i),
$$

where $s_i$ is current shed stock. Extra sales are introduced only when $C>F$. Feed, fertilizer, and unplaced livestock needed by the block are protected first; remaining products are sold in descending visible-price order. Sales are moved before purchases so the funding is available when the market queue executes.


## Execution invariants

The action list is aligned to the live hand count. Shed projection now accounts for `PICKUP` as well as same-turn `DROP` and `PLACE`:

$$
\widehat{s}_i
=s_i-\operatorname{pickup}_i
+\operatorname{drop}_i
+\operatorname{place}_i,
$$

with every addition capped by remaining shed capacity. Sell clamping then processes market orders sequentially, credits earlier `BUY_PRODUCT` orders, and retains zero-quantity sell placeholders so later market race positions do not shift.


## Scope of the revision

The route tapes, route thresholds, weed rule, 99-item day-close reserve, dead-stock logic, and exact turn-718 settlement are preserved. In an on-track state with sufficient funding and the expected number of hands, the candidate emits the same actions as its parent. The new layers intervene only when the live state no longer satisfies those execution assumptions.

Running the code cell writes a standalone `main.py` and a deterministic `submission.tar.gz` containing only that file.


In [ ]:
from pathlib import Path
import base64
import gzip
import hashlib
import io
import tarfile

EXPECTED_SOURCE_SHA256 = "1590394cf30c8989a3e1cfdb027d89b5ccd97c2fb4cdc14959a15555b2b2361a"
EXPECTED_ARCHIVE_SHA256 = "4690ff7c922b2cc51d8791d7da447359e090d75976b7e73c605e17191c8d38cd"
MAIN_B64 = '...'

source = base64.b64decode(MAIN_B64)
assert hashlib.sha256(source).hexdigest() == EXPECTED_SOURCE_SHA256
compile(source, "main.py", "exec")
Path("main.py").write_bytes(source)

raw_tar = io.BytesIO()
with tarfile.open(fileobj=raw_tar, mode="w") as package:
    info = tarfile.TarInfo("main.py")
    info.size = len(source)
    info.mode = 0o644
    info.mtime = 0
    info.uid = info.gid = 0
    info.uname = info.gname = ""
    package.addfile(info, io.BytesIO(source))

with Path("submission.tar.gz").open("wb") as raw:
    with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as compressed:
        compressed.write(raw_tar.getvalue())

assert hashlib.sha256(Path("submission.tar.gz").read_bytes()).hexdigest() == EXPECTED_ARCHIVE_SHA256
with tarfile.open("submission.tar.gz", "r:gz") as package:
    assert package.getnames() == ["main.py"]
    assert package.extractfile("main.py").read() == source

print({
    "agent": "Shape the Shop | Safe Execution Hybrid",
    "main_py_sha256": EXPECTED_SOURCE_SHA256,
    "archive_sha256": EXPECTED_ARCHIVE_SHA256,
    "archive_members": ["main.py"],
})
